# Introduction: Exploring Peru with Python
in this notebook, we will create interactive maps and explore how Python geoespatial libraries can work together. We will use locations and examples from Peru.
We will begin with geogradev , then extend the workflow with other tools such as Folium, GeoPandas, Rasterio, and ipyleaflet in later examples.

## 1. Create your first interactive map
let's star in Lima. The coordinates below are expressed as latitude and longitude in decimal degrees.

In [ ]:
import geogradev.foliumap as geogradev

# Approximate geographic coordinates of central lima
lima = [-12.0464, -77.0428]

# Create an interactive map
m = geogradev.Map(center=lima, zoom=14, height="600px")

m

## 2. Exploring Geographic locations
Let's explore different geographical locations across Peru.
In this exercise, we will use geographic coordinates to move between different regions and visualize their landscapes.

In [ ]:
# Geographic locations across Peru

locations = {
    "Lima": [-12.0464, -77.0428],
    "Cusco": [-13.5319, -71.9675],
    "Arequipa": [-16.4090, -71.5375],
    "Trujillo": [-8.1091, -79.0215],
    "Iquitos": [-3.7437, -73.2516],
    "Puno": [-15.8402, -70.0219],
    "Chiclayo": [-6.7714, -79.8400],
    "Piura": [-5.1945, -80.6328],
    "Huaraz": [-9.5270, -77.5280],
    "Tarapoto": [-6.4860, -76.3540],
}

In [ ]:
# Select a location
location = "Huaraz"  # always be careful with the spelling of the location name

# Create an interactive map
m = geogradev.Map(center=locations[location], zoom=14, height="600px")

m

### Explore the map

Change the `location` to explore other geographical locations.

**Try it:**

- Visualize Lima and Paracas.
- Explore the Andes through Huaraz and Ollantaytambo.
- Explore the transition to the Amazon region through Moyobamba.

Observe how the geographic landscape changes between regions

## 3. Working with different Basemaps

Basemaps provide the geographic context for spatial data.
Different basemaps can highlight different characteristics of the landscape, such as roads, settlements, terrain, and setellite imagery.
Now we will explore different basemap styles using 'geogradev'.

In [ ]:
# Explore satellite imagery
import folium

m = geogradev.Map(center=locations["Lima"], zoom=12, tiles=None, height="600px")

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Esri World Imagery",
).add_to(m)

m

In [ ]:
# Explore a topographic basemap
m = geogradev.Map(
    center=locations["Huaraz"], zoom=15, basemap="Esri.WorldTopoMap", height="600px"
)

m

### Explore and Compare

Observe how the same geographic area is represented using different basemaps.

**Try it:**

1. Change the location from Lima to Chiclayo
2. Compare satellite imagery with the topographic map.
3. Explore how roads, settlements, and terrain are represented.
4. Select the basemap that best suits you mapping objective.

**Question** Which basemap would you choose to study urban expansion, and which one would you use to examine mountainous terrain?


## 4. Exploring Geographic data with GeoPandas

GeoPandas allows us to work with geographic vector data using Python.
In this exercise, we will create a GeoDataFrame containing geographic locations across Peru and visualize them on an interactive map.

In [ ]:
# We have to import the geopandas library to work with geospatial data in Python.
# Geopandas extends the datatypes used by pandas to allow spatial operations on geometric types.
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

In [ ]:
# Define geographic regions
regions = {
    "Lima": "Coast",
    "Cusco": "Andes",
    "Arequipa": "Andes",
    "Trujillo": "Coast",
    "Iquitos": "Amazon",
    "Puno": "Andes",
    "Chiclayo": "Coast",
    "Piura": "Coast",
    "Huaraz": "Andes",
    "Tarapoto": "Amazon",
}

# Create a DataFrame
df = pd.DataFrame(
    [
        {
            "Location": name,
            "Latitude": coordinates[0],
            "Longitude": coordinates[1],
            "Region": regions[name],
        }
        for name, coordinates in locations.items()
    ]
)

df

In [ ]:
# Create point geometries
geometry = [Point(lon, lat) for lat, lon in zip(df["Latitude"], df["Longitude"])]

# Create GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

gdf

In [ ]:
m = gdf.explore(
    column="Region",
    tooltip=["Location", "Latitude", "Longitude", "Region"],
    popup=True,
    tiles=None,
    cmap="Set1",
    legend=True,
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Esri World Street Map",
).add_to(m)

m

In [ ]:
print("Coordinate Reference System:", gdf.crs)
print("Geometry type:", gdf.geom_type.unique())
print("Number of locations:", len(gdf))
print("Geographic bounds:", gdf.total_bounds)

### Explore the geographic data

**Try it:**

1. Add another location to the `locations` dictionary.
2. Assign its corresponding geographic region.
3. Observe how the GeoDataFrame and interactive map are updated.
4. Compare the distribution of locations across Peru.

**Question:** Why is it important to define a Coordinate Reference System (CRS) when working with geographic data?

## 5. Exploring raster data with Rasterio

Raster data represent geographic information as a grid of cells, where each cell contains a specific value.
in this exercise, we will use rasterio to explore a DEM covering Peru.
we will inspect it's spatial properties, calculate elevation statistics, and visualize the terrain.


In [ ]:
# Define the raster path
dem_path = "../data/dem_peru_1km.tif"

# Open the raster dataset
with rasterio.open(dem_path) as src:

    dem = src.read(1)

    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Dimensions:", src.width, "x", src.height)
    print("Number of bands:", src.count)
    print("Data type:", src.dtypes)
    print("NoData:", src.nodata)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Mask NoData values
dem_masked = np.ma.masked_equal(dem, 0)

print("Minimum elevation:", dem_masked.min())
print("Maximum elevation:", dem_masked.max())
print("Mean elevation:", dem_masked.mean())

In [ ]:
plt.figure(figsize=(10, 8))

plt.imshow(dem_masked, cmap="terrain")

plt.colorbar(label="Elevation (meters)")

plt.title("Digital Elevation Model of Peru")
plt.xlabel("Column")
plt.ylabel("Row")

plt.show()

## 6. Mini Mapping Challenge: Exploring Peru

In this final exercise, we will combine the geospatial tools introduced throughout this notebook.
we will use GeoPandas to manage geographic locations, rasterio to access elevation data, and geogradev with folium to create an interactive map.
our goal is to build a geographic visualization of Peru combining city locations and terrain information.

In [ ]:
from rasterio.warp import calculate_default_transform, reproject, Resampling

dst_crs = "EPSG:3857"

with rasterio.open(dem_path) as src:
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )

    dem_web = np.full((height, width), np.nan, dtype=np.float32)

    reproject(
        source=rasterio.band(src, 1),
        destination=dem_web,
        src_transform=src.transform,
        src_crs=src.crs,
        src_nodata=src.nodata,
        dst_transform=transform,
        dst_crs=dst_crs,
        dst_nodata=np.nan,
        resampling=Resampling.bilinear,
    )

print("Reprojected DEM shape:", dem_web.shape)
print("Destination CRS:", dst_crs)

In [ ]:
from matplotlib.colors import Normalize

valid = dem_web[np.isfinite(dem_web)]

vmin, vmax = np.percentile(valid, [2, 98])

norm = Normalize(vmin=vmin, vmax=vmax)

rgba = plt.cm.terrain(norm(np.nan_to_num(dem_web, nan=vmin)))

rgba = (rgba * 255).astype(np.uint8)

rgba[~np.isfinite(dem_web)] = [0, 0, 0, 0]

print("RGBA image shape:", rgba.shape)

In [ ]:
from rasterio.warp import transform_bounds
from folium.raster_layers import ImageOverlay

west, south, east, north = transform_bounds(
    dst_crs,
    "EPSG:4326",
    transform.c,
    transform.f + height * transform.e,
    transform.c + width * transform.a,
    transform.f,
)

m = geogradev.Map(center=[-9.5, -75], zoom=5, tiles=None, height="650px")

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Esri World Street Map",
).add_to(m)

ImageOverlay(
    image=rgba,
    bounds=[[south, west], [north, east]],
    opacity=0.65,
    name="Digital Elevation Model",
).add_to(m)

for _, row in gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        popup=row["Location"],
        tooltip=row["Location"],
        color="red",
        fill=True,
        fill_opacity=0.8,
    ).add_to(m)

folium.LayerControl().add_to(m)

m